In [7]:
from get_lat_lon import get_lat_lon
import xarray as xr
import numpy as np
import os

def read_nc_dd():
    #deseason_detrend, except tp
    sst=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/sst.nc")
    sm=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/sm.nc")
    lai=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/lai.nc")
    t2m=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/t2m.nc")
    ssrd=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/ssrd.nc")
    bdod=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/static/bdod.nc")
    sand=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/static/sand.nc")
    silt=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/static/silt.nc")
    clay=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/static/clay.nc")
    tp=xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/tp2.nc")
    return [lai,sst,t2m,ssrd,bdod,sand,silt,clay,tp,sm]


def get_upwind_weights_mask(x_cell,lat_values,lon_values,mask,period=str("yr")):
# Please type below within the brackets the period of interest (yr, Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec) 
# Please choose the land grid cell index which represents the region of interest
# The index can be gained via the array coordinates_land_grid_cells[:,0:8683] and ranges from 0 to 8683
# array_ccordinates_land_grid_cells [0,:] -> latitudes; array_ccordinates_land_grid_cells [1,:] -> longitudes 
# Land cells referring to the index 0 to 63 (represent land cells at the northern boundary of the considered grid) were set to zero and produce thus no results 
    cell_longitudes=np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/cell_longitudes.npy") 
    cell_latitudes=np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/cell_latitudes.npy")
    path=str("/Net/Groups/BGI/scratch/fhuang/utrack")   
    os.chdir(path+r"/Dataset/Matrices/Land_cell_to_grid(Era_Int_2001_2018)") 
    string_for_loading=str("Era_Int_2001_2018_matrix_"+period+str(".npy"))
    Era_Int_2001_2018_cell = np.load(string_for_loading)


    Considered_cell=Era_Int_2001_2018_cell[:,[x_cell]] 
    Reshaped_array_cell = Considered_cell.reshape(107,240)
    
    # Defining x,y and z values
    longitudes_cells=np.tile(cell_longitudes,(107,1))
    latitudes_cells=np.transpose(np.tile(cell_latitudes,(240,1)))     
    values_yr_percent_cell=Reshaped_array_cell/np.sum(Reshaped_array_cell)

    #flip the shape
    res=values_yr_percent_cell.copy()
    res[:,:120]=values_yr_percent_cell[:,120:]
    res[:,120:]=values_yr_percent_cell[:,:120]

    res_df=xr.DataArray(res,dims=( "latitude","longitude"),  coords={"latitude":lat_values,"longitude": lon_values})
    res_df.data[~mask.data] = np.nan
    res_out=res_df/np.nansum(res_df)
    return res_out

def generate_xy(inv, input_list):
    x_cells=np.load("sa_x_cell.npy")
    considered_cells=np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/considered_cells.npy")
    month_list=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    cell_longitudes=np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/cell_longitudes.npy") 
    cell_latitudes=np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/cell_latitudes.npy")
    lon_values=(cell_longitudes+ 0.75)-180
    lat_values=cell_latitudes+ 0.75
    mask=xr.open_dataarray(('/Net/Groups/BGI/scratch/fhuang/utrack/uta_res/mask.nc'))
    
    
    lai,sst,t2m,ssrd,bdod,sand,silt,clay,tp,sm=input_list

    nsample=x_cells.shape[0]
    input_data=np.full((nsample,216,inv*2+1,inv*2+1,11),np.nan)
    y_data=np.full((nsample,216),np.nan)
    k=0
    

    for i in range(nsample):
        x_cell=x_cells[i]
        lati,loni=considered_cells[:,x_cell]
        loni=loni-120
        lat,lon=get_lat_lon(x_cell)
        lati=lati-1
    
        up=lati-inv
        down=lati+inv+1
        left=loni-inv
        right=loni+inv+1
        
        
        
        laiv=lai.data[:,up:down,left:right]
        sstv=sst.data[:,up:down,left:right]
        t2mv=t2m.data[:,up:down,left:right]
        rnv=ssrd.data[:,up:down,left:right]
    
        bdodv=bdod.data[up:down,left:right]#0--surface density
        sandv=sand.data[up:down,left:right]
        siltv=silt.data[up:down,left:right]
        clayv=clay.data[up:down,left:right]
        tpv=tp.data[:,up:down,left:right]
        smv=sm.data[:,up:down,left:right]
        
        smv_i=sm.data[:,lati,loni]
        input_data[i,:,:,:,0]=laiv
        input_data[i,:,:,:,1]=sstv
        input_data[i,:,:,:,2]=t2mv
        input_data[i,:,:,:,3]=rnv
     
        input_data[i,:,:,:,4]=bdodv
        input_data[i,:,:,:,5]=sandv
        input_data[i,:,:,:,6]=siltv
        input_data[i,:,:,:,7]=clayv
        input_data[i,:,:,:,8]=tpv
        
        mi=0
        for year in range(18):
            for month in month_list:
                weights_pic=get_upwind_weights_mask(x_cell,lat_values,lon_values,mask,period=str(month))
                input_data[i,mi,:,:,9]=weights_pic[up:down,left:right]
                mi=mi+1
        
        
        input_data[i,:,:,:,10]=smv
        y_data[i,:]=smv_i
        print(x_cell,i,lati,loni,lat,lon)
        
    return input_data,y_data

def remove_y_nan(x,y,x_cells):

    nan_idx=[]
    for i in range(x.shape[0]):
        yi=y[i,:]
        if np.sum(np.isnan(yi))==216:
            nan_idx.append(i)
            
    select_cell_clean=np.delete(x_cells, nan_idx)
    x_clean=np.delete(x,nan_idx,axis=0)
    y_clean=np.delete(y,nan_idx,axis=0)
    
    return x_clean,y_clean,select_cell_clean

In [ ]:
input_data,y_data=generate_xy(inv=10, input_list=read_nc_dd())